# TCD to TCN 변환

> 교통카드 데이터(TCD)를 통행체인 네트워크(TCN) 형식으로 변환하는 파이프라인

**목적**: 원시 TCD 데이터를 정제하여 유효한 통행 기록(TCN)으로 변환
- 동일 O-D 제거
- 좌표 누락 데이터 제거  
- 500m 미만 단거리 통행 필터링
- parquet 형식으로 저장

---
## 1. 설정

In [ ]:
%load_ext autoreload
%autoreload

from module.tcd_to_tcn_route import process_multiple_dates
from module.tcd_to_tcn_route import *

import warnings

warnings.filterwarnings('ignore')

---
## 2. 데이터 변환 실행

In [ ]:
# dates = ['20250217', '20250218', '20250219', '20250220', '20250221', '20250222', '20250223']
dates = [ '20250221']

results = process_multiple_dates(
    dates,
    base_path='C:/Folder/Research/0. DATA/tcd_2025_parquet',
    output_dir='../../data/tcn',
    split_round_trip=False,
    sig_path='../../data/shp/sig',
    ctprvn_path='../../data/shp/ctprvn',
)

---
## 3. 결과 확인

In [ ]:
# 결과 확인
for date, path in results.items():
    tcn = pd.read_parquet(path)
    hidden = (tcn['숨겨진환승횟수'] > 0).sum()
    print(f'{date}: {len(tcn):,} trips, 숨겨진환승 {hidden:,}건 ({hidden/len(tcn)*100:.1f}%)')
    del tcn

---
## 4. 단계별 TCD → TCN 변환 과정
> `process_date` 함수의 내부 파이프라인을 단계별로 실행하여 각 과정을 확인

### 4-1. 데이터 로드

In [72]:
# 단일 날짜 설정
date = '20250217'
base_path = 'C:/Folder/Research/0. DATA/tcd_2025_parquet'

# TCD, ROUT, ROUTESTTN 로드
loader = TCDLoader(base_path)
tcd, route, routesttn = loader.load_data(date)

print(f"TCD: {len(tcd):,} records")
print(f"ROUT: {len(route):,} routes")
print(f"ROUTESTTN: {len(routesttn):,} route-station records")

TCD: 17,826,894 records
ROUT: 4,010 routes
ROUTESTTN: 257,579 route-station records


### 4-2. 전처리: 무효 통행 제거

In [36]:
# 승하차 동일 정류장 제거
mask_same_stop = tcd["승차정류장ID(정산사업자)"] == tcd["하차정류장ID(정산사업자)"]
invalid_trips = tcd.loc[mask_same_stop, ["가상카드번호", "트랜잭션ID"]].drop_duplicates()

tcd_clean = tcd.merge(
    invalid_trips, on=["가상카드번호", "트랜잭션ID"],
    how="left", indicator=True
).query("_merge == 'left_only'").drop(columns="_merge")
print(f"동일 O-D 제거 후: {len(tcd_clean):,} records")

# 정류장ID NA인 통행 제거
sttn_cols = ["승차정류장ID(정산사업자)", "하차정류장ID(정산사업자)"]
mask_na_sttn = tcd_clean[sttn_cols].isna().any(axis=1)
invalid_na_sttn = tcd_clean.loc[mask_na_sttn, ["가상카드번호", "트랜잭션ID"]].drop_duplicates()

tcd_clean = tcd_clean.merge(
    invalid_na_sttn, on=["가상카드번호", "트랜잭션ID"],
    how="left", indicator=True
).query("_merge == 'left_only'").drop(columns="_merge")
print(f"NA 정류장ID 제거 후: {len(tcd_clean):,} records (제거: {len(invalid_na_sttn):,} trips)")

동일 O-D 제거 후: 17,430,369 records
NA 정류장ID 제거 후: 17,213,257 records (제거: 187,318 trips)


In [43]:
route[route['노선ID'] == '325']

,운행일자,정산사 ID,정산지역코드,노선ID,노선명(long),노선명(short),교통수단유형,총운행거리,정류장수
961,20250217,8,11100,325,GTX-A,수도권광역급행철도에이,G,0,0


In [56]:
tcd_clean[tcd_clean['노선ID(정산사업자)'] == '325']

,운행일자,정산사 ID,인련번호,가상카드번호,정산지역코드,카드구분코드,차량ID(국토부표준),차량ID(정산사업자),차량등록번호,운행출발일시,...,승차정류장ID(정산사업자),하차정류장ID(국토부표준),하차정류장ID(정산사업자),하차일시,트랜잭션ID,환승건수,사용자구분코드,이용자수,이용거리,탑승시간
4323,20250217,8,1,++1u0OoH1qRgGn1EZoZ5A77YZ0aOpSsrmlTvv/AwefM=,11100,2,NaN,NaN,None,NaN,...,9001,NaN,9005.0,2.025022e+13,74,0,1,1,26200,1579
4706,20250217,8,4,++7K+APaedX8Xyyxg15Hwvq1bXByWJLf9IeO6qCphcM=,11100,2,NaN,NaN,None,NaN,...,9004,NaN,9000.0,2.025022e+13,98,1,1,1,22900,1785
4817,20250217,8,1,++93BTmt3gxYTz7txBNWHs9CPGtLR/LCrrM0Uvi5FYA=,11100,2,NaN,NaN,None,NaN,...,9001,NaN,9005.0,2.025022e+13,4,0,1,1,26200,1807
4818,20250217,8,2,++93BTmt3gxYTz7txBNWHs9CPGtLR/LCrrM0Uvi5FYA=,11100,2,NaN,NaN,None,NaN,...,9005,NaN,9001.0,2.025022e+13,5,0,1,1,26200,1772
5809,20250217,8,1,++OY869fpM6LbnOCIWp572PB+YJ2RpkUL9WpSUHwVeo=,11100,2,NaN,NaN,None,NaN,...,9010,NaN,9008.0,2.025022e+13,28,0,1,1,22100,1539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17427739,20250217,8,1,zzOAeodyST83LhmsmiipTVA1mTxn3ypM8r1qAyO1mVo=,11100,2,NaN,NaN,None,NaN,...,9008,NaN,9010.0,2.025022e+13,50,0,1,1,22100,2201
17428412,20250217,8,4,zzYgcm2hsWKcUgCtCHxKcq3VoqrhCNgkFmfoNhR6Ajc=,11100,2,NaN,NaN,None,NaN,...,9002,NaN,9005.0,2.025022e+13,9,2,1,1,19300,1216
17428559,20250217,8,2,zzapkRQJBTJNApkwSekybogWJkwEBJvibDQ/lOhKfZY=,11100,2,NaN,NaN,None,NaN,...,9004,NaN,9005.0,2.025022e+13,48,1,1,1,9400,664
17429539,20250217,8,1,zzrYrgKBVyIhvH6P959yvzKVVPRoUJNYTzjNmlBuGrs=,11100,2,NaN,NaN,None,NaN,...,9010,NaN,9008.0,2.025022e+13,58,0,1,1,22100,1290


### 4-3. TCD 전처리 + ROUTESTTN 병합

In [ ]:
preprocessor = TCDPreprocessor()

# TCD 전처리 (컬럼명 변경, 타입 변환)
tcd_pre = preprocessor.preprocess_tcd(tcd_clean)

# ROUTESTTN 전처리
routesttn_pre = preprocessor.preprocess_routesttn(routesttn)

# TCD + ROUTESTTN 병합 (노선ID + 정류장ID 기반 좌표 매칭)
tcd_merged = preprocessor.merge_tcd_routesttn(tcd_pre, routesttn_pre)

print(f"GTX 통행 건수 : {len(tcd_merged[tcd_merged['노선ID'] == '325'])}")

In [65]:
routesttn[routesttn['노선ID'] == '325']


,운행일자,정산사 ID,정산지역코드,노선ID,노선명(short),교통수단유형,정류장순서,정류장 ID,정류장 명칭,정류장 X 좌표,정류장 Y 좌표,정류장 ARS번호,누적거리(m),구간거리(m)
61976,20250217,8,11100,325,GTX-A,T,1,9000,운정중앙,37.716067,126.728156,~,0,0
61977,20250217,8,11100,325,GTX-A,T,2,9001,킨텍스,37.665252,126.748253,~,0,0
61978,20250217,8,11100,325,GTX-A,T,3,9002,대곡,37.632401,126.810449,~,0,0
61979,20250217,8,11100,325,GTX-A,T,4,9004,연신내,37.618926,126.920643,~,0,0
61980,20250217,8,11100,325,GTX-A,T,5,9005,서울,37.555506,126.972403,~,0,0
61981,20250217,8,11100,325,GTX-A,T,6,9006,삼성,37.509598,127.062776,~,0,0
61982,20250217,8,11100,325,GTX-A,T,7,9007,수서,37.486263,127.103003,~,0,0
61983,20250217,8,11100,325,GTX-A,T,8,9008,성남,37.394974,127.120526,~,0,0
61984,20250217,8,11100,325,GTX-A,T,9,9009,구성,37.299134,127.103864,~,0,0
61985,20250217,8,11100,325,GTX-A,T,10,9010,동탄,37.199924,127.095485,~,0,0


In [62]:
routesttn_pre[routesttn_pre['노선ID'] == '325']

,운행일자,정산사 ID,정산지역코드,노선ID,노선명(short),교통수단유형,정류장순서,정류장 ID,정류장 명칭,정류장 X 좌표,정류장 Y 좌표,누적거리(m),구간거리(m)
61976,20250217,8,11100,325,GTX-A,T,1,9000,운정중앙,37.716067,126.728156,0,0
61977,20250217,8,11100,325,GTX-A,T,2,9001,킨텍스,37.665252,126.748253,0,0
61978,20250217,8,11100,325,GTX-A,T,3,9002,대곡,37.632401,126.810449,0,0
61979,20250217,8,11100,325,GTX-A,T,4,9004,연신내,37.618926,126.920643,0,0
61980,20250217,8,11100,325,GTX-A,T,5,9005,서울,37.555506,126.972403,0,0
61981,20250217,8,11100,325,GTX-A,T,6,9006,삼성,37.509598,127.062776,0,0
61982,20250217,8,11100,325,GTX-A,T,7,9007,수서,37.486263,127.103003,0,0
61983,20250217,8,11100,325,GTX-A,T,8,9008,성남,37.394974,127.120526,0,0
61984,20250217,8,11100,325,GTX-A,T,9,9009,구성,37.299134,127.103864,0,0
61985,20250217,8,11100,325,GTX-A,T,10,9010,동탄,37.199924,127.095485,0,0


In [54]:
tcd_merged[tcd_merged['노선ID'] == '325']

,운행일자,정산사 ID,인련번호,가상카드번호,정산지역코드,카드구분코드,차량ID(정산사업자),차량등록번호,운행출발일시,운행종료일시,...,승차노선명,승차정류장순서,승차누적거리,하차정류장 명칭,하차정류장 X 좌표,하차정류장 Y 좌표,하차교통수단유형,하차노선명,하차정류장순서,하차누적거리
4143,20250217,8,1,++1u0OoH1qRgGn1EZoZ5A77YZ0aOpSsrmlTvv/AwefM=,11100,2,NaN,None,NaN,NaN,...,GTX-A,2.0,0.0,서울,37.555506,126.972403,T,GTX-A,5.0,0.0
4519,20250217,8,4,++7K+APaedX8Xyyxg15Hwvq1bXByWJLf9IeO6qCphcM=,11100,2,NaN,None,NaN,NaN,...,GTX-A,4.0,0.0,운정중앙,37.716067,126.728156,T,GTX-A,1.0,0.0
4630,20250217,8,1,++93BTmt3gxYTz7txBNWHs9CPGtLR/LCrrM0Uvi5FYA=,11100,2,NaN,None,NaN,NaN,...,GTX-A,2.0,0.0,서울,37.555506,126.972403,T,GTX-A,5.0,0.0
4631,20250217,8,2,++93BTmt3gxYTz7txBNWHs9CPGtLR/LCrrM0Uvi5FYA=,11100,2,NaN,None,NaN,NaN,...,GTX-A,5.0,0.0,킨텍스,37.665252,126.748253,T,GTX-A,2.0,0.0
5607,20250217,8,1,++OY869fpM6LbnOCIWp572PB+YJ2RpkUL9WpSUHwVeo=,11100,2,NaN,None,NaN,NaN,...,GTX-A,10.0,0.0,성남,37.394974,127.120526,T,GTX-A,8.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17210661,20250217,8,1,zzOAeodyST83LhmsmiipTVA1mTxn3ypM8r1qAyO1mVo=,11100,2,NaN,None,NaN,NaN,...,GTX-A,8.0,0.0,동탄,37.199924,127.095485,T,GTX-A,10.0,0.0
17211327,20250217,8,4,zzYgcm2hsWKcUgCtCHxKcq3VoqrhCNgkFmfoNhR6Ajc=,11100,2,NaN,None,NaN,NaN,...,GTX-A,3.0,0.0,서울,37.555506,126.972403,T,GTX-A,5.0,0.0
17211471,20250217,8,2,zzapkRQJBTJNApkwSekybogWJkwEBJvibDQ/lOhKfZY=,11100,2,NaN,None,NaN,NaN,...,GTX-A,4.0,0.0,서울,37.555506,126.972403,T,GTX-A,5.0,0.0
17212442,20250217,8,1,zzrYrgKBVyIhvH6P959yvzKVVPRoUJNYTzjNmlBuGrs=,11100,2,NaN,None,NaN,NaN,...,GTX-A,10.0,0.0,성남,37.394974,127.120526,T,GTX-A,8.0,0.0


In [57]:
# 통행 구분 코드 생성
tcd_merged = preprocessor.create_trip_id(tcd_merged)

# 좌표 매칭 현황
o_matched = tcd_merged['승차정류장 X 좌표'].notna().sum()
d_matched = tcd_merged['하차정류장 X 좌표'].notna().sum()
print(f"좌표 매칭: 승차 {o_matched:,}/{len(tcd_merged):,} ({o_matched/len(tcd_merged)*100:.1f}%), "
      f"하차 {d_matched:,}/{len(tcd_merged):,} ({d_matched/len(tcd_merged)*100:.1f}%)")

좌표 매칭: 승차 17,138,646/17,213,257 (99.6%), 하차 17,121,036/17,213,257 (99.5%)


### 4-4. 좌표 무효 통행 제거

In [58]:
coord_cols = ['승차정류장 X 좌표', '승차정류장 Y 좌표', '하차정류장 X 좌표', '하차정류장 Y 좌표']

# 좌표 0인 통행 제거
mask_zero = (tcd_merged[coord_cols] == 0).any(axis=1)
invalid_zero = tcd_merged.loc[mask_zero, ["가상카드번호", "트랜잭션ID"]].drop_duplicates()

tcd_merged = tcd_merged.merge(
    invalid_zero, on=["가상카드번호", "트랜잭션ID"],
    how="left", indicator=True
).query("_merge == 'left_only'").drop(columns="_merge")
print(f"좌표 0 제거 후: {len(tcd_merged):,} records (제거: {len(invalid_zero):,} trips)")

# 좌표 NA인 통행 제거
mask_na_coords = tcd_merged[coord_cols].isna().any(axis=1)
invalid_na_trips = tcd_merged.loc[mask_na_coords, ["가상카드번호", "트랜잭션ID"]].drop_duplicates()

tcd_final = tcd_merged.merge(
    invalid_na_trips, on=["가상카드번호", "트랜잭션ID"],
    how="left", indicator=True
).query("_merge == 'left_only'").drop(columns="_merge")
print(f"좌표 NA 제거 후: {len(tcd_final):,} records (제거: {len(invalid_na_trips):,} trips)")

좌표 0 제거 후: 17,183,306 records (제거: 19,097 trips)
좌표 NA 제거 후: 16,911,748 records (제거: 154,356 trips)


In [ ]:
tcd_final[tcd_final['노선ID'] == '325']

,운행일자,정산사 ID,인련번호,가상카드번호,정산지역코드,카드구분코드,차량ID(정산사업자),차량등록번호,운행출발일시,운행종료일시,...,승차누적거리,하차정류장 명칭,하차정류장 X 좌표,하차정류장 Y 좌표,하차교통수단유형,하차노선명,하차정류장순서,하차누적거리,환승횟수_재계산,구분코드
4000,20250217,8,1,++1u0OoH1qRgGn1EZoZ5A77YZ0aOpSsrmlTvv/AwefM=,11100,2,NaN,None,NaN,NaN,...,0.0,서울,37.555506,126.972403,T,GTX-A,5.0,0.0,0,++1u0OoH1qRgGn1EZoZ5A77YZ0aOpSsrmlTvv/AwefM=_74
4376,20250217,8,4,++7K+APaedX8Xyyxg15Hwvq1bXByWJLf9IeO6qCphcM=,11100,2,NaN,None,NaN,NaN,...,0.0,운정중앙,37.716067,126.728156,T,GTX-A,1.0,0.0,1,++7K+APaedX8Xyyxg15Hwvq1bXByWJLf9IeO6qCphcM=_98
4487,20250217,8,1,++93BTmt3gxYTz7txBNWHs9CPGtLR/LCrrM0Uvi5FYA=,11100,2,NaN,None,NaN,NaN,...,0.0,서울,37.555506,126.972403,T,GTX-A,5.0,0.0,0,++93BTmt3gxYTz7txBNWHs9CPGtLR/LCrrM0Uvi5FYA=_4
4488,20250217,8,2,++93BTmt3gxYTz7txBNWHs9CPGtLR/LCrrM0Uvi5FYA=,11100,2,NaN,None,NaN,NaN,...,0.0,킨텍스,37.665252,126.748253,T,GTX-A,2.0,0.0,0,++93BTmt3gxYTz7txBNWHs9CPGtLR/LCrrM0Uvi5FYA=_5
5464,20250217,8,1,++OY869fpM6LbnOCIWp572PB+YJ2RpkUL9WpSUHwVeo=,11100,2,NaN,None,NaN,NaN,...,0.0,성남,37.394974,127.120526,T,GTX-A,8.0,0.0,0,++OY869fpM6LbnOCIWp572PB+YJ2RpkUL9WpSUHwVeo=_28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17180715,20250217,8,1,zzOAeodyST83LhmsmiipTVA1mTxn3ypM8r1qAyO1mVo=,11100,2,NaN,None,NaN,NaN,...,0.0,동탄,37.199924,127.095485,T,GTX-A,10.0,0.0,0,zzOAeodyST83LhmsmiipTVA1mTxn3ypM8r1qAyO1mVo=_50
17181380,20250217,8,4,zzYgcm2hsWKcUgCtCHxKcq3VoqrhCNgkFmfoNhR6Ajc=,11100,2,NaN,None,NaN,NaN,...,0.0,서울,37.555506,126.972403,T,GTX-A,5.0,0.0,2,zzYgcm2hsWKcUgCtCHxKcq3VoqrhCNgkFmfoNhR6Ajc=_9
17181523,20250217,8,2,zzapkRQJBTJNApkwSekybogWJkwEBJvibDQ/lOhKfZY=,11100,2,NaN,None,NaN,NaN,...,0.0,서울,37.555506,126.972403,T,GTX-A,5.0,0.0,1,zzapkRQJBTJNApkwSekybogWJkwEBJvibDQ/lOhKfZY=_48
17182494,20250217,8,1,zzrYrgKBVyIhvH6P959yvzKVVPRoUJNYTzjNmlBuGrs=,11100,2,NaN,None,NaN,NaN,...,0.0,성남,37.394974,127.120526,T,GTX-A,8.0,0.0,0,zzrYrgKBVyIhvH6P959yvzKVVPRoUJNYTzjNmlBuGrs=_58


### 4-5. 지하철 환승 그래프 구축 + TCN 변환

In [ ]:
# 지하철/GTX 노선 간 환승 그래프 구축
transfer_graph = SubwayTransferGraph(routesttn)

# TCN 변환 (왕복 분리 없음)
converter = TCDtoTCNConverter(
    split_round_trip=False,
    transfer_graph=transfer_graph
)
tcn = converter.convert(tcd_final)

# NaN 제거
tcn = tcn.dropna().reset_index(drop=True)
print(f"\nFinal TCN: {len(tcn):,} trips")
tcn.head()

### 4-6. 지역 필터링 + 저장

In [ ]:
# GTX 영향권 지역 필터링 (출발지 기준)
sig_path = '../../data/shp/sig'
ctprvn_path = '../../data/shp/ctprvn'

tcn_filtered = TCDPreprocessor.filter_gtx_origin(
    tcn,
    sig_path=sig_path,
    ctprvn_path=ctprvn_path,
    o_lon_col='승차정류장 Y 좌표',
    o_lat_col='승차정류장 X 좌표'
)
print(f"지역 필터링 후: {len(tcn_filtered):,} trips")

# 저장
import os
output_path = f'../../data/tcn/{date}/TCN_{date}_route.parquet'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
tcn_filtered.to_parquet(output_path)
print(f"저장 완료: {output_path}")

### 4-7. 결과 통계

In [ ]:
# 교통수단 카테고리별 통행 분포
print(f"=== {date} TCN 통계 ===")
print(f"총 통행 수: {len(tcn_filtered):,}")
print(f"\n교통수단 카테고리별 분포:")
print(tcn_filtered['transport_category'].value_counts().to_string())

# 숨겨진 환승 통계
hidden = (tcn_filtered['숨겨진환승횟수'] > 0).sum()
print(f"\n숨겨진환승: {hidden:,}건 ({hidden/len(tcn_filtered)*100:.1f}%)")